# Train the universal telemetry anomaly model

This notebook trains **one final model**: an `XGBClassifier` trained with CUDA and served on the laptop CPU.

The public PSM and SMD datasets have different numbers and meanings of metrics. Each series is therefore converted into the same 16 causal temporal features (robust deviation, short/long change, and fast/slow drift). The final classifier can then be calibrated for the five metrics emitted by the local checkout service.

Success criteria:

- train on public labeled telemetry with a Colab GPU;
- validate on an SMD machine excluded from development training;
- produce one small `foundation.ubj` artifact;
- keep local inference single-threaded and fast enough for a 10-second sampling interval.


## Data and evaluation boundary

- [PSM](https://github.com/eBay/RANSynCoders) is downloaded from the eBay repository at a pinned commit. Its sample data is CC BY 4.0.
- [SMD](https://github.com/NetManAIOps/OmniAnomaly) is downloaded from the OmniAnomaly repository at a pinned commit. SMD entities stay separate during feature construction.
- The final SMD entity is held out while the development model is checked. The final artifact is then fitted once on all selected public entities.
- Local service data is used only to learn five median/IQR reference values and a normal-score threshold. The tree model is not refitted on the laptop.

This notebook runs the full dataset: PSM plus all 28 SMD machines. No manual dataset upload is required.


In [ ]:
# Keep Colab's compatible NumPy, pandas, and scikit-learn stack intact.
# Install only the model library; pip keeps already-compatible Colab dependencies.
!pip -q install --upgrade-strategy only-if-needed "xgboost==3.1.3"

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY = "https://github.com/Cyaside/local-ml-service-monitor.git"
CHECKOUT = Path("/content/local-ml-service-monitor")
if CHECKOUT.exists():
    shutil.rmtree(CHECKOUT)
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(CHECKOUT)], check=True)
sys.path.insert(0, str(CHECKOUT / "src"))

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0 or "GPU" not in gpu.stdout:
    raise RuntimeError("GPU is unavailable. In Colab choose Runtime > Change runtime type > T4 GPU, then run all again.")
print(gpu.stdout.strip())

import numpy, pandas, sklearn, xgboost
print({"numpy": numpy.__version__, "pandas": pandas.__version__, "sklearn": sklearn.__version__, "xgboost": xgboost.__version__})
import json
probe = xgboost.XGBClassifier(n_estimators=1, max_depth=1, tree_method="hist", device="cuda")
probe.fit(numpy.array([[0.0], [1.0], [2.0], [3.0]]), numpy.array([0, 0, 1, 1]))
actual_device = json.loads(probe.get_booster().save_config())["learner"]["generic_param"]["device"]
if not actual_device.startswith("cuda"):
    raise RuntimeError(f"XGBoost did not use the GPU (device={actual_device}).")
print("XGBoost GPU smoke check passed:", actual_device)
from service_monitor.ml.public_training import train_public_foundation
print("Source ready:", CHECKOUT)


## Configuration

The final run uses PSM and all 28 SMD entities. The notebook requests a T4 GPU runtime and XGBoost uses CUDA for both the held-out fit and the final fit.


In [ ]:
PROFILE = "full"
CACHE_DIR = Path("/content/public-telemetry-cache")
OUTPUT_DIR = Path(f"/content/telemetry-foundation-{PROFILE}")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

print({"profile": PROFILE, "cache": str(CACHE_DIR), "output": str(OUTPUT_DIR)})


## Train the single foundation model

This is the only long-running cell. It downloads the selected public data, builds causal temporal features, performs the cross-domain check, fits the final model, and writes the artifact. You can leave Colab running and return later.


In [ ]:
metadata = train_public_foundation(
    cache_dir=CACHE_DIR,
    output_dir=OUTPUT_DIR,
    profile=PROFILE,
    device="cuda",
)
metadata


## Inspect the held-out result and artifact size

Average precision is the most useful point-level metric here because anomalies are rare. Event recall answers whether each anomalous interval was noticed at least once. These public results validate the learned pattern; the separate local held-out test still decides whether the calibrated service model is acceptable.


In [ ]:
import pandas as pd

summary = pd.Series({
    **metadata["validation"],
    "training_rows": metadata["training_rows"],
    "positive_rows": metadata["positive_rows"],
    "training_seconds": metadata["training_seconds"],
    "artifact_mib": (OUTPUT_DIR / "foundation.ubj").stat().st_size / 1024**2,
})
display(summary.to_frame("value"))


## Download the completed artifact

The folder contains the model, metadata, checksum, and completion marker. It is zipped and downloaded automatically when training finishes. No Drive mount or manual data upload is needed.


In [ ]:
from google.colab import files

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Training complete. Downloading:", archive)
files.download(archive)


## Calibrate and test on the laptop

After copying the foundation artifact into the repository, use the already-recorded normal baseline. Calibration computes local reference statistics and a conservative normal-score threshold; it does not refit the classifier.

```powershell
uv run service-monitor calibrate `
  --foundation models/foundation-public-v1 `
  --baseline data/exports/train-baseline-20260919T132930.csv `
  --out models/checkout-universal-v1

uv run service-monitor evaluate `
  --model models/checkout-universal-v1 `
  --input data/exports/test-20260919T180539.csv `
  --report reports/checkout-universal-v1-test.json
```

Only promote the artifact for live monitoring after the untouched local test has acceptable incident recall and false alarms.
